In [0]:
%run ../00-common/01.environment-config

In [0]:
%python
control_table=f"{catalog_name}.{control_schema}.batch_control"

In [0]:
%python
from pyspark.sql import functions as F

# Get batch folders from landing
landing_batches = sorted([
  file.name.rstrip('/') 
  for file in dbutils.fs.ls(landing_folder_path)
  if file.isDir()
])

# read tracked batches
if spark.catalog.tableExists(control_table):
  tracked_batches = [
    row.batch_id 
    for row in (
        spark.table(control_table).filter(F.col("status").isin("completed", "in_progress")).select('batch_id').distinct().collect()
    )
  ]
else:
  tracked_batches = []
  
# Identify earliest unprocessed batch
new_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = new_batches[0] if len(new_batches) > 0 else None

print(f"Landing Batches: {landing_batches}")
print(f"Tracked Batches: {tracked_batches}")
print(f"New Batches: {new_batches}")
print(f"Next Batch to process: {next_batch}")

if next_batch is None:
  dbutils.jobs.taskValues.set(key="p_batch_id",value="")
  dbutils.jobs.taskValues.set(key="has_batch",value=False)
else:
  dbutils.jobs.taskValues.set(key="p_batch_id",value=next_batch)
  dbutils.jobs.taskValues.set(key="has_batch",value=True)